In [ ]:
# METHOD 2 — RESNET18 TRANSFER LEARNING

!pip -q install torch torchvision scikit-learn pandas matplotlib seaborn pillow



import os
import random
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 65)
print("METHOD 2 — RESNET18 TRANSFER LEARNING")
print("=" * 65)

print("PyTorch version :", torch.__version__)
print("Torchvision     :", __import__("torchvision").__version__)
print("Device          :", device)

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print("GPU             : Not available")

print("\nSetup complete.")

In [ ]:
# UPLOAD AND VERIFY THE 100 BMP IMAGES

from google.colab import files
import os

# Create image directory
IMAGE_DIR = "/content/images"
os.makedirs(IMAGE_DIR, exist_ok=True)

print("Please select ALL 100 BMP images.")
uploaded = files.upload()

# Save uploaded files
for filename, data in uploaded.items():
    save_path = os.path.join(IMAGE_DIR, filename)

    with open(save_path, "wb") as f:
        f.write(data)

print("\n" + "=" * 60)
print("UPLOAD COMPLETE")
print("=" * 60)

# Check BMP files
bmp_files = sorted(
    [
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(".bmp")
    ],
    key=lambda x: int(
        os.path.splitext(x)[0].split("_")[1]
    )
)

print("BMP images found:", len(bmp_files))

# Check missing files
expected_files = [f"s_{i}.bmp" for i in range(1, 101)]

missing_files = [
    f for f in expected_files
    if f not in bmp_files
]

if missing_files:
    print("\nMissing files:")
    for f in missing_files:
        print(" ", f)
else:
    print("All 100 expected images are present.")

print("\nFirst 10 images:")
for f in bmp_files[:10]:
    print(" ", f)

print("\nLast 10 images:")
for f in bmp_files[-10:]:
    print(" ", f)

Please select ALL 100 BMP images.


In [ ]:
# CREATE AND VERIFY LABEL TABLE

# Labels for all 100 images
LABELS = {
    # Original 37 images
    "s_1.bmp": "good",
    "s_2.bmp": "good",
    "s_3.bmp": "good",
    "s_4.bmp": "good",
    "s_5.bmp": "good",
    "s_6.bmp": "good",
    "s_7.bmp": "good",
    "s_8.bmp": "good",
    "s_9.bmp": "fail",
    "s_10.bmp": "fail",
    "s_11.bmp": "good",
    "s_12.bmp": "good",
    "s_13.bmp": "fail",
    "s_14.bmp": "fail",
    "s_15.bmp": "good",
    "s_16.bmp": "good",
    "s_17.bmp": "fail",
    "s_18.bmp": "good",
    "s_19.bmp": "fail",
    "s_20.bmp": "fail",
    "s_21.bmp": "fail",
    "s_22.bmp": "fail",
    "s_23.bmp": "fail",
    "s_24.bmp": "good",
    "s_25.bmp": "good",
    "s_26.bmp": "fail",
    "s_27.bmp": "good",
    "s_28.bmp": "fail",
    "s_29.bmp": "fail",
    "s_30.bmp": "fail",
    "s_31.bmp": "good",
    "s_32.bmp": "good",
    "s_33.bmp": "fail",
    "s_34.bmp": "good",
    "s_35.bmp": "good",
    "s_36.bmp": "fail",
    "s_37.bmp": "fail",

    # New images
    "s_38.bmp": "good",
    "s_39.bmp": "good",
    "s_40.bmp": "good",
    "s_41.bmp": "good",
    "s_42.bmp": "good",
    "s_43.bmp": "good",
    "s_44.bmp": "good",
    "s_45.bmp": "good",
    "s_46.bmp": "good",
    "s_47.bmp": "good",
    "s_48.bmp": "good",
    "s_49.bmp": "good",
    "s_50.bmp": "good",
    "s_51.bmp": "good",
    "s_52.bmp": "good",
    "s_53.bmp": "good",
    "s_54.bmp": "good",
    "s_55.bmp": "good",
    "s_56.bmp": "good",
    "s_57.bmp": "good",
    "s_58.bmp": "good",
    "s_59.bmp": "good",
    "s_60.bmp": "good",
    "s_61.bmp": "good",
    "s_62.bmp": "good",
    "s_63.bmp": "good",
    "s_64.bmp": "good",
    "s_65.bmp": "good",
    "s_66.bmp": "good",
    "s_67.bmp": "good",
    "s_68.bmp": "good",
    "s_69.bmp": "good",
    "s_70.bmp": "good",
    "s_71.bmp": "good",
    "s_72.bmp": "good",

    "s_73.bmp": "fail",
    "s_74.bmp": "fail",
    "s_75.bmp": "fail",
    "s_76.bmp": "fail",
    "s_77.bmp": "fail",
    "s_78.bmp": "fail",
    "s_79.bmp": "fail",
    "s_80.bmp": "fail",
    "s_81.bmp": "fail",
    "s_82.bmp": "fail",
    "s_83.bmp": "fail",
    "s_84.bmp": "fail",
    "s_85.bmp": "fail",
    "s_86.bmp": "fail",
    "s_87.bmp": "fail",
    "s_88.bmp": "fail",
    "s_89.bmp": "fail",
    "s_90.bmp": "fail",
    "s_91.bmp": "fail",
    "s_92.bmp": "fail",
    "s_93.bmp": "fail",
    "s_94.bmp": "fail",
    "s_95.bmp": "fail",
    "s_96.bmp": "fail",
    "s_97.bmp": "fail",
    "s_98.bmp": "fail",
    "s_99.bmp": "fail",
    "s_100.bmp": "fail"
}



label_df = pd.DataFrame(
    [
        {
            "filename": filename,
            "label": LABELS[filename]
        }
        for filename in expected_files
    ]
)



print("=" * 60)
print("LABEL VERIFICATION")
print("=" * 60)

print("\nTotal labels:", len(label_df))

print("\nClass distribution:")
print(label_df["label"].value_counts())


# Check against expected numbers


good_count = (label_df["label"] == "good").sum()
fail_count = (label_df["label"] == "fail").sum()

print("\nExpected:")
print("Good:", 55)
print("Fail:", 45)

print("\nActual:")
print("Good:", good_count)
print("Fail:", fail_count)

if good_count == 55 and fail_count == 45:
    print("\n✓ LABEL CHECK PASSED")
else:
    print("\n✗ LABEL CHECK FAILED")


print("\nComplete label table:")
display(label_df)

In [ ]:
#  VISUAL CHECK OF THE DATASET

import math

# Number of images to display
N_DISPLAY = 20

# Select first 20 images
display_df = label_df.iloc[:N_DISPLAY]

# Create figure
fig, axes = plt.subplots(
    4,
    5,
    figsize=(18, 14)
)

axes = axes.flatten()

for ax, (_, row) in zip(axes, display_df.iterrows()):

    filename = row["filename"]
    label = row["label"]

    image_path = os.path.join(
        IMAGE_DIR,
        filename
    )

    # Read image
    image = Image.open(image_path).convert("RGB")

    # Display
    ax.imshow(image)

    # Title
    ax.set_title(
        f"{filename}\nLabel: {label.upper()}",
        fontsize=10
    )

    ax.axis("off")

# Hide unused axes
for ax in axes[len(display_df):]:
    ax.axis("off")

fig.suptitle(
    "Dataset Visual Check — First 20 Images",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# RESNET18 DATASET & IMAGE TRANSFORMS

from torchvision import transforms
from PIL import Image


IMG_SIZE = 224

# Training transformations

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Small realistic variations
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),

    transforms.ToTensor(),

    # ImageNet normalization for pretrained ResNet18
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validation / test transformation

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("=" * 60)
print("RESNET18 DATASET PREPARATION")
print("=" * 60)

print("Image size :", IMG_SIZE, "x", IMG_SIZE)
print("Channels   : RGB")
print("Training augmentation : Horizontal flip + ±10° rotation")
print("Validation augmentation : None")
print("Normalization : ImageNet")
print()
print("Dataset ready for 5-fold stratified cross-validation.")

In [ ]:
# RESNET18 TRANSFER-LEARNING MODEL

import torch
import torch.nn as nn
from torchvision import models

# Create pretrained ResNet18

resnet18 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Replace final classification layer
# 1000 ImageNet classes -> 2 classes

num_features = resnet18.fc.in_features

resnet18.fc = nn.Linear(
    num_features,
    2
)

# Move model to GPU/CPU

resnet18 = resnet18.to(device)

# Class mapping

class_names = ["good", "fail"]

print("=" * 60)
print("RESNET18 TRANSFER LEARNING MODEL")
print("=" * 60)

print("Architecture       : ResNet18")
print("Pretrained weights : ImageNet")
print("Original classes   : 1000")
print("Final classes      : 2")
print("Classes            :", class_names)
print("Final layer        :", resnet18.fc)
print("Device             :", device)

print("\n✓ ResNet18 model ready.")

In [ ]:
#  CREATE RESNET18 LABEL ARRAY
# ALL 100 IMAGES — EXPLICIT LABEL DICTIONARY

import numpy as np
import pandas as pd

# Create filenames

filenames = [f"s_{i}.bmp" for i in range(1, 101)]

# Labels for ALL 100 images

labels_dict = {

    1: "good",
    2: "good",
    3: "good",
    4: "good",
    5: "good",
    6: "good",
    7: "good",
    8: "good",
    9: "fail",
    10: "fail",

    11: "good",
    12: "good",
    13: "fail",
    14: "fail",
    15: "good",
    16: "good",
    17: "fail",
    18: "good",
    19: "fail",
    20: "fail",

    21: "fail",
    22: "fail",
    23: "fail",
    24: "good",
    25: "good",
    26: "fail",
    27: "good",
    28: "fail",
    29: "fail",
    30: "fail",

    31: "good",
    32: "good",
    33: "fail",
    34: "good",
    35: "good",
    36: "fail",
    37: "fail",

    38: "good",
    39: "good",
    40: "good",
    41: "good",
    42: "good",
    43: "good",
    44: "good",
    45: "good",
    46: "good",
    47: "good",
    48: "good",
    49: "good",
    50: "good",

    51: "good",
    52: "good",
    53: "good",
    54: "good",
    55: "good",
    56: "good",
    57: "good",
    58: "good",
    59: "good",
    60: "good",
    61: "good",
    62: "good",
    63: "good",
    64: "good",
    65: "good",
    66: "good",
    67: "good",
    68: "good",
    69: "good",
    70: "good",
    71: "good",
    72: "good",

    73: "fail",
    74: "fail",
    75: "fail",
    76: "fail",
    77: "fail",
    78: "fail",
    79: "fail",
    80: "fail",
    81: "fail",
    82: "fail",
    83: "fail",
    84: "fail",
    85: "fail",
    86: "fail",
    87: "fail",
    88: "fail",
    89: "fail",
    90: "fail",

    91: "fail",
    92: "fail",
    93: "fail",
    94: "fail",
    95: "fail",
    96: "fail",
    97: "fail",
    98: "fail",
    99: "fail",
    100: "fail"
}

# Create dataframe

labels_df = pd.DataFrame({
    "filename": filenames,
    "label": [labels_dict[i] for i in range(1, 101)]
})

# Convert labels to numerical classes
# 0 = Good
# 1 = Fail

labels = np.array([
    0 if label == "good" else 1
    for label in labels_df["label"]
])

# Verification


print("=" * 60)
print("RESNET18 LABEL ARRAY")
print("=" * 60)

print("Total images :", len(labels))
print("Good        :", np.sum(labels == 0))
print("Fail        :", np.sum(labels == 1))

print("\nExpected: 55 Good / 45 Fail")

assert len(labels) == 100
assert np.sum(labels == 0) == 55
assert np.sum(labels == 1) == 45

print("✓ LABEL CHECK PASSED")


# Display all 100 labels

display(labels_df)

In [ ]:
# VERIFY RESNET18 LABELS


import numpy as np
import pandas as pd


print("=" * 60)
print("CELL 6B — RESNET18 LABEL VERIFICATION")
print("=" * 60)

print("\nNumber of entries in labels_dict:", len(labels_dict))

assert len(labels_dict) == 100

# Check that keys 1–100 all exist
expected_keys = set(range(1, 101))
actual_keys = set(labels_dict.keys())

missing_keys = expected_keys - actual_keys
extra_keys = actual_keys - expected_keys

print("Missing label numbers:", missing_keys)
print("Extra label numbers  :", extra_keys)

assert len(missing_keys) == 0
assert len(extra_keys) == 0

# ------------------------------------------------------------
# Check that only good/fail labels are present
# ------------------------------------------------------------

valid_labels = {"good", "fail"}

invalid_labels = {
    value for value in labels_dict.values()
    if value not in valid_labels
}

print("Invalid labels:", invalid_labels)

assert len(invalid_labels) == 0

# ------------------------------------------------------------
# Create numerical target array
# 0 = Good
# 1 = Fail
# ------------------------------------------------------------

labels = np.array([
    0 if labels_dict[i] == "good" else 1
    for i in range(1, 101)
])

# ------------------------------------------------------------
# Class counts
# ------------------------------------------------------------

good_count = np.sum(labels == 0)
fail_count = np.sum(labels == 1)

print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

print("Total images :", len(labels))
print("Good         :", good_count)
print("Fail         :", fail_count)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(labels) == 100
assert good_count == 55
assert fail_count == 45

print("\n✓ 100 LABELS FOUND")
print("✓ ALL LABELS ARE VALID")
print("✓ 55 GOOD / 45 FAIL")
print("✓ NUMERICAL LABEL ARRAY CREATED")
print("✓ CELL 6B PASSED")

# ------------------------------------------------------------
# Show label mapping
# ------------------------------------------------------------

verification_df = pd.DataFrame({
    "image_number": range(1, 101),
    "filename": [f"s_{i}.bmp" for i in range(1, 101)],
    "label": [
        labels_dict[i]
        for i in range(1, 101)
    ],
    "class_number": labels
})

print("\nComplete verification table:")
display(verification_df)

In [ ]:

# CELL 7 — RESNET18 5-FOLD STRATIFIED CROSS-VALIDATION


import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix




IMAGE_DIR = "/content/images"

N_FOLDS = 5
EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 0.0001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("RESNET18 — 5-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 60)

print("Images       :", len(filenames))
print("Folds        :", N_FOLDS)
print("Epochs/fold  :", EPOCHS)
print("Batch size   :", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Device       :", DEVICE)



# TRANSFORMS


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



# CUSTOM DATASET

class SealantDataset(Dataset):

    def __init__(self, filenames, labels, transform=None):

        self.filenames = filenames
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):

        filename = self.filenames[idx]

        image_path = os.path.join(
            IMAGE_DIR,
            filename
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = int(self.labels[idx])

        return image, label


# ============================================================
# STRATIFIED K-FOLD
# ============================================================

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

all_predictions = np.zeros(len(labels), dtype=int)
all_true_labels = np.array(labels)

fold_accuracies = []


# ============================================================
# TRAIN EACH FOLD
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    skf.split(filenames, labels), start=1
):

    print("\n" + "=" * 60)
    print(f"FOLD {fold}/{N_FOLDS}")
    print("=" * 60)

    # --------------------------------------------------------
    # Datasets
    # --------------------------------------------------------

    train_dataset = SealantDataset(
        [filenames[i] for i in train_idx],
        labels[train_idx],
        transform=train_transform
    )

    val_dataset = SealantDataset(
        [filenames[i] for i in val_idx],
        labels[val_idx],
        transform=val_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    # --------------------------------------------------------
    # Create fresh ResNet18
    # --------------------------------------------------------

    model = models.resnet18(
        weights=models.ResNet18_Weights.DEFAULT
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        2
    )

    model = model.to(DEVICE)

    # --------------------------------------------------------
    # Loss + optimizer
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, targets in train_loader:

            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                targets
            )

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(
                outputs,
                1
            )

            total += targets.size(0)
            correct += (
                predicted == targets
            ).sum().item()

        train_accuracy = correct / total

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, targets in val_loader:

                images = images.to(DEVICE)
                targets = targets.to(DEVICE)

                outputs = model(images)

                _, predicted = torch.max(
                    outputs,
                    1
                )

                val_total += targets.size(0)

                val_correct += (
                    predicted == targets
                ).sum().item()

        val_accuracy = val_correct / val_total

        print(
            f"Epoch {epoch+1:02d}/{EPOCHS} | "
            f"Loss: {running_loss/len(train_loader):.4f} | "
            f"Train Acc: {train_accuracy:.3f} | "
            f"Val Acc: {val_accuracy:.3f}"
        )

    # ========================================================
    # FINAL VALIDATION PREDICTIONS
    # ========================================================

    model.eval()

    fold_predictions = []

    with torch.no_grad():

        for images, targets in val_loader:

            images = images.to(DEVICE)

            outputs = model(images)

            _, predicted = torch.max(
                outputs,
                1
            )

            fold_predictions.extend(
                predicted.cpu().numpy()
            )

    fold_predictions = np.array(
        fold_predictions
    )

    all_predictions[val_idx] = fold_predictions

    fold_accuracy = accuracy_score(
        labels[val_idx],
        fold_predictions
    )

    fold_accuracies.append(
        fold_accuracy
    )

    print(
        f"\nFold {fold} validation accuracy: "
        f"{fold_accuracy:.3f}"
    )


# ============================================================
# OVERALL OUT-OF-FOLD RESULTS
# ============================================================

overall_accuracy = accuracy_score(
    all_true_labels,
    all_predictions
)

cm = confusion_matrix(
    all_true_labels,
    all_predictions
)

print("\n")
print("=" * 60)
print("RESNET18 — FINAL 5-FOLD RESULTS")
print("=" * 60)

print(
    f"Overall accuracy : {overall_accuracy:.3f}"
)

print(
    "\nFold accuracies:"
)

for i, acc in enumerate(
    fold_accuracies,
    start=1
):

    print(
        f"Fold {i}: {acc:.3f}"
    )

print(
    f"\nMean fold accuracy: "
    f"{np.mean(fold_accuracies):.3f}"
)

print(
    f"Std fold accuracy : "
    f"{np.std(fold_accuracies):.3f}"
)

print("\nConfusion matrix:")
print(cm)

print("\n✓ RESNET18 TRAINING COMPLETE")

In [ ]:
# ============================================================
# CELL 8 — RESNET18 COMPLETE PERFORMANCE METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("RESNET18 — 5-FOLD OUT-OF-FOLD PERFORMANCE")
print("=" * 60)

# ------------------------------------------------------------
# Calculate metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_true_labels,
    all_predictions
)

precision = precision_score(
    all_true_labels,
    all_predictions,
    pos_label=1
)

recall = recall_score(
    all_true_labels,
    all_predictions,
    pos_label=1
)

f1 = f1_score(
    all_true_labels,
    all_predictions,
    pos_label=1
)

macro_f1 = f1_score(
    all_true_labels,
    all_predictions,
    average="macro"
)

# ------------------------------------------------------------
# Print overall results
# ------------------------------------------------------------

print(f"Accuracy  : {accuracy:.3f}")
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"F1-score  : {f1:.3f}")
print(f"Macro-F1  : {macro_f1:.3f}")

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        all_true_labels,
        all_predictions,
        target_names=["Good", "Fail"],
        digits=3
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_true_labels,
    all_predictions
)

print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

print(cm)

# ------------------------------------------------------------
# Verify expected total
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)

print("Total predictions :", len(all_predictions))
print("Correct           :", np.sum(
    all_true_labels == all_predictions
))
print("Incorrect         :", np.sum(
    all_true_labels != all_predictions
))

assert len(all_true_labels) == 100
assert len(all_predictions) == 100

print("\n✓ 100 OUT-OF-FOLD PREDICTIONS VERIFIED")
print("✓ RESNET18 METRICS CALCULATED")

In [ ]:
# ============================================================
# CELL 9 — RESNET18 CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ------------------------------------------------------------
# Create confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_true_labels,
    all_predictions
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Good", "Fail"],
    yticklabels=["Good", "Fail"],
    annot_kws={"size": 16}
)

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)

plt.title(
    "ResNet18 — 5-Fold Out-of-Fold Confusion Matrix",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_path = "/content/resnet18_confusion_matrix.png"

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(output_path)

print("\nConfusion matrix:")
print(cm)

In [ ]:
# ============================================================
# CELL 10 — RESNET18 100-IMAGE OUT-OF-FOLD PREDICTIONS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Check required arrays
# ------------------------------------------------------------

print("=" * 60)
print("RESNET18 — 100 IMAGE OUT-OF-FOLD PREDICTIONS")
print("=" * 60)

print("True labels :", len(all_true_labels))
print("Predictions :", len(all_predictions))

# ------------------------------------------------------------
# Create filenames
# ------------------------------------------------------------

filenames = [f"s_{i}.bmp" for i in range(1, 101)]

# ------------------------------------------------------------
# Convert numerical labels to text
# 0 = Good
# 1 = Fail
# ------------------------------------------------------------

true_text = [
    "good" if x == 0 else "fail"
    for x in all_true_labels
]

pred_text = [
    "good" if x == 0 else "fail"
    for x in all_predictions
]

# ------------------------------------------------------------
# Correct / incorrect
# ------------------------------------------------------------

correct = np.array(all_true_labels) == np.array(all_predictions)

# ------------------------------------------------------------
# Create dataframe
# ------------------------------------------------------------

resnet18_predictions = pd.DataFrame({
    "filename": filenames,
    "true_label": true_text,
    "predicted_label": pred_text,
    "correct": correct
})

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total = len(resnet18_predictions)
correct_count = int(correct.sum())
incorrect_count = total - correct_count

accuracy = correct_count / total

print("\n============================================================")
print("PREDICTION SUMMARY")
print("============================================================")

print("Total images :", total)
print("Correct      :", correct_count)
print("Incorrect    :", incorrect_count)
print(f"Accuracy     : {accuracy:.3f}")

# ------------------------------------------------------------
# Show first 20
# ------------------------------------------------------------

print("\nFirst 20 predictions:")

display(
    resnet18_predictions.head(20)
)

# ------------------------------------------------------------
# Show incorrectly classified images
# ------------------------------------------------------------

print("\n============================================================")
print("INCORRECTLY CLASSIFIED IMAGES")
print("============================================================")

display(
    resnet18_predictions[
        resnet18_predictions["correct"] == False
    ]
)

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

output_csv = "/content/resnet18_100_image_predictions.csv"

resnet18_predictions.to_csv(
    output_csv,
    index=False
)

print("\nSaved:")
print(output_csv)

print("\n✓ 100 IMAGE PREDICTION TABLE CREATED")

In [ ]:
# ============================================================
# CELL 11 — RESNET18 100-IMAGE VISUAL RESULT SHEETS
# 4 SHEETS × 25 IMAGES
# ============================================================

import os
import cv2
import matplotlib.pyplot as plt
import numpy as np

OUTPUT_DIR = "/content/resnet18_result_sheets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_DIR = "/content/images"


# ------------------------------------------------------------
# Create visual for one image
# ------------------------------------------------------------

def create_resnet18_visual(filename):

    image_path = os.path.join(IMAGE_DIR, filename)

    image = cv2.imread(image_path)

    if image is None:
        return None

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    return image


# ------------------------------------------------------------
# Create 4 result sheets
# ------------------------------------------------------------

for sheet_number in range(4):

    start = sheet_number * 25
    end = min(start + 25, len(resnet18_predictions))

    subset = resnet18_predictions.iloc[start:end]

    fig, axes = plt.subplots(
        5,
        5,
        figsize=(20, 20)
    )

    axes = axes.flatten()

    for ax, (_, row) in zip(
        axes,
        subset.iterrows()
    ):

        filename = row["filename"]

        visual = create_resnet18_visual(filename)

        if visual is not None:
            ax.imshow(visual)

        true_label = row["true_label"].upper()
        pred_label = row["predicted_label"].upper()

        if row["correct"]:
            symbol = "✓"
        else:
            symbol = "✗"

        ax.set_title(
            f"{filename}\n"
            f"TRUE: {true_label} | PRED: {pred_label} {symbol}",
            fontsize=9
        )

        ax.axis("off")

    # --------------------------------------------------------
    # Hide unused axes
    # --------------------------------------------------------

    for ax in axes[len(subset):]:
        ax.axis("off")

    # --------------------------------------------------------
    # Main title
    # --------------------------------------------------------

    fig.suptitle(
        f"ResNet18 — Sealant Bead Inspection Results "
        f"({start + 1}–{end})",
        fontsize=18,
        fontweight="bold"
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_path = os.path.join(
        OUTPUT_DIR,
        f"resnet18_results_{start + 1}_{end}.png"
    )

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print(f"Saved: {output_path}")


print("\n" + "=" * 60)
print("RESNET18 VISUAL RESULT SHEETS COMPLETE")
print("=" * 60)
print("Total sheets : 4")
print("Images       : 100")
print("Images/sheet : 25")
print("Output folder:", OUTPUT_DIR)

In [ ]:
# ============================================================
# CELL 12 — RESNET18 TRAINING CURVES
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = "/content/resnet18_training_curves"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# RECORDED RESULTS FROM RESNET18 CELL 7
# ============================================================

epochs = np.arange(1, 11)

# ------------------------------------------------------------
# Fold 1
# ------------------------------------------------------------

fold1_train_acc = [
    0.713, 0.863, 0.950, 0.950, 0.975,
    0.963, 0.988, 0.975, 0.988, 0.988
]

fold1_val_acc = [
    0.700, 0.650, 0.650, 0.600, 0.650,
    0.650, 0.700, 0.700, 0.650, 0.700
]

fold1_loss = [
    0.5023, 0.3113, 0.1389, 0.1025, 0.1050,
    0.0885, 0.0501, 0.0825, 0.0547, 0.0465
]

# ------------------------------------------------------------
# Fold 2
# ------------------------------------------------------------

fold2_train_acc = [
    0.700, 0.900, 0.875, 0.950, 0.925,
    0.963, 0.912, 0.988, 0.988, 1.000
]

fold2_val_acc = [
    0.900, 0.950, 0.900, 0.850, 0.900,
    0.950, 0.950, 0.850, 0.900, 1.000
]

fold2_loss = [
    0.5732, 0.2382, 0.2829, 0.1721, 0.1862,
    0.0894, 0.1438, 0.0428, 0.0788, 0.0300
]

# ------------------------------------------------------------
# Fold 3
# ------------------------------------------------------------

fold3_train_acc = [
    0.713, 0.900, 0.900, 0.950, 0.975,
    0.975, 0.988, 0.988, 0.950, 0.975
]

fold3_val_acc = [
    0.800, 0.850, 0.850, 0.750, 0.800,
    0.850, 0.800, 0.800, 0.750, 0.800
]

fold3_loss = [
    0.5263, 0.2898, 0.2615, 0.1396, 0.0867,
    0.0595, 0.0415, 0.0218, 0.0701, 0.0494
]

# ------------------------------------------------------------
# Fold 4
# ------------------------------------------------------------

fold4_train_acc = [
    0.713, 0.900, 0.925, 0.988, 0.950,
    0.963, 1.000, 1.000, 1.000, 1.000
]

fold4_val_acc = [
    0.650, 0.800, 0.750, 0.700, 0.700,
    0.700, 0.700, 0.700, 0.800, 0.700
]

fold4_loss = [
    0.5070, 0.2655, 0.2043, 0.0654, 0.1287,
    0.1073, 0.0159, 0.0280, 0.0441, 0.0191
]

# ------------------------------------------------------------
# Fold 5
# ------------------------------------------------------------

fold5_train_acc = [
    0.575, 0.912, 0.925, 0.975, 1.000,
    0.912, 0.900, 0.925, 0.912, 0.975
]

fold5_val_acc = [
    0.850, 0.850, 0.850, 0.850, 0.950,
    0.950, 0.900, 0.900, 0.850, 0.900
]

fold5_loss = [
    0.7286, 0.2294, 0.1796, 0.1212, 0.0340,
    0.2237, 0.1708, 0.1589, 0.1980, 0.0920
]

# ------------------------------------------------------------
# Put all folds together
# ------------------------------------------------------------

train_acc_all = [
    fold1_train_acc,
    fold2_train_acc,
    fold3_train_acc,
    fold4_train_acc,
    fold5_train_acc
]

val_acc_all = [
    fold1_val_acc,
    fold2_val_acc,
    fold3_val_acc,
    fold4_val_acc,
    fold5_val_acc
]

loss_all = [
    fold1_loss,
    fold2_loss,
    fold3_loss,
    fold4_loss,
    fold5_loss
]

# ============================================================
# FIGURE 1 — TRAINING AND VALIDATION ACCURACY
# ============================================================

plt.figure(figsize=(10, 7))

for i in range(5):

    plt.plot(
        epochs,
        np.array(train_acc_all[i]) * 100,
        marker="o",
        label=f"Fold {i+1} — Training"
    )

    plt.plot(
        epochs,
        np.array(val_acc_all[i]) * 100,
        marker="x",
        linestyle="--",
        label=f"Fold {i+1} — Validation"
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.title(
    "ResNet18 — Training and Validation Accuracy",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(epochs)
plt.ylim(40, 105)
plt.grid(alpha=0.3)
plt.legend(fontsize=8, ncol=2)

plt.tight_layout()

accuracy_path = os.path.join(
    OUTPUT_DIR,
    "resnet18_training_validation_accuracy.png"
)

plt.savefig(
    accuracy_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(accuracy_path)


# ============================================================
# FIGURE 2 — TRAINING LOSS
# ============================================================

plt.figure(figsize=(10, 7))

for i in range(5):

    plt.plot(
        epochs,
        loss_all[i],
        marker="o",
        label=f"Fold {i+1}"
    )

plt.xlabel("Epoch")
plt.ylabel("Training Loss")

plt.title(
    "ResNet18 — Training Loss Across 5 Folds",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(epochs)
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

loss_path = os.path.join(
    OUTPUT_DIR,
    "resnet18_training_loss.png"
)

plt.savefig(
    loss_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(loss_path)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("RESNET18 TRAINING CURVES COMPLETE")
print("=" * 60)

print("Accuracy plot:")
print(accuracy_path)

print("\nLoss plot:")
print(loss_path)

In [ ]:
# ============================================================
# CELL 10 — RESNET18 ERROR ANALYSIS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import os

# ------------------------------------------------------------
# Load the ResNet18 prediction results
# ------------------------------------------------------------

prediction_file = "/content/resnet18_100_image_predictions.csv"

pred_df = pd.read_csv(prediction_file)

print("=" * 60)
print("RESNET18 — ERROR ANALYSIS")
print("=" * 60)

print("Total images :", len(pred_df))

# ------------------------------------------------------------
# Find incorrectly classified images
# ------------------------------------------------------------

incorrect_df = pred_df[
    pred_df["correct"] == False
].copy()

print("Incorrect images :", len(incorrect_df))

print("\nIncorrectly classified images:")
print(incorrect_df.to_string(index=False))

# ------------------------------------------------------------
# Error types
# ------------------------------------------------------------

good_to_fail = incorrect_df[
    (incorrect_df["true_label"] == "good") &
    (incorrect_df["predicted_label"] == "fail")
]

fail_to_good = incorrect_df[
    (incorrect_df["true_label"] == "fail") &
    (incorrect_df["predicted_label"] == "good")
]

print("\n" + "=" * 60)
print("ERROR BREAKDOWN")
print("=" * 60)

print("Good → Fail :", len(good_to_fail))
print("Fail → Good :", len(fail_to_good))

# ------------------------------------------------------------
# Save error analysis
# ------------------------------------------------------------

error_file = "/content/resnet18_error_analysis.csv"

incorrect_df.to_csv(
    error_file,
    index=False
)

print("\nSaved:")
print(error_file)

print("\n✓ RESNET18 ERROR ANALYSIS COMPLETE")

In [ ]:
# ============================================================
# CELL 11 — VISUALISE RESNET18 MISCLASSIFIED IMAGES
# ============================================================

import os
import math
import matplotlib.pyplot as plt
from PIL import Image

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

IMAGE_DIR = "/content/images"

# ------------------------------------------------------------
# Get incorrect filenames
# ------------------------------------------------------------

error_files = incorrect_df["filename"].tolist()

print("=" * 60)
print("RESNET18 — MISCLASSIFIED IMAGE VISUALISATION")
print("=" * 60)

print("Number of incorrect images:", len(error_files))

# ------------------------------------------------------------
# Create figure
# ------------------------------------------------------------

cols = 3
rows = math.ceil(len(error_files) / cols)

plt.figure(figsize=(15, rows * 4))

# ------------------------------------------------------------
# Plot each incorrect image
# ------------------------------------------------------------

for i, filename in enumerate(error_files):

    image_path = os.path.join(IMAGE_DIR, filename)

    image = Image.open(image_path).convert("RGB")

    true_label = incorrect_df.loc[
        incorrect_df["filename"] == filename,
        "true_label"
    ].values[0]

    predicted_label = incorrect_df.loc[
        incorrect_df["filename"] == filename,
        "predicted_label"
    ].values[0]

    plt.subplot(rows, cols, i + 1)

    plt.imshow(image)

    plt.title(
        f"{filename}\n"
        f"True: {true_label.upper()} | "
        f"Pred: {predicted_label.upper()}"
    )

    plt.axis("off")

plt.suptitle(
    "ResNet18 — Misclassified Images",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

# ------------------------------------------------------------
# Save figure
# ------------------------------------------------------------

output_file = "/content/resnet18_misclassified_images.png"

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(output_file)

print("\n✓ MISCLASSIFIED IMAGE VISUALISATION COMPLETE")

In [ ]:
# ============================================================
# CELL 12 — RESNET18 GRAD-CAM INTERPRETABILITY
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

print("=" * 60)
print("RESNET18 — GRAD-CAM ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# Check that the trained model exists
# ------------------------------------------------------------

if "model" not in globals():
    raise NameError(
        "Trained ResNet18 model not found. "
        "Please run the ResNet18 training cell first."
    )

model.eval()
model.to(device)

# ------------------------------------------------------------
# Target layer
# ResNet18 final convolutional block
# ------------------------------------------------------------

target_layer = model.layer4[-1]

activations = []
gradients = []

# ------------------------------------------------------------
# Forward hook
# ------------------------------------------------------------

def forward_hook(module, input, output):
    activations.append(output)

# ------------------------------------------------------------
# Backward hook
# ------------------------------------------------------------

def backward_hook(module, grad_input, grad_output):
    gradients.append(grad_output[0])

forward_handle = target_layer.register_forward_hook(
    forward_hook
)

backward_handle = target_layer.register_full_backward_hook(
    backward_hook
)

# ------------------------------------------------------------
# Select 4 representative misclassified images
# ------------------------------------------------------------

fail_to_good = incorrect_df[
    (incorrect_df["true_label"] == "fail") &
    (incorrect_df["predicted_label"] == "good")
]["filename"].tolist()

good_to_fail = incorrect_df[
    (incorrect_df["true_label"] == "good") &
    (incorrect_df["predicted_label"] == "fail")
]["filename"].tolist()

selected_files = (
    fail_to_good[:2] +
    good_to_fail[:2]
)

print("\nSelected images:")
for f in selected_files:
    print(" -", f)

# ------------------------------------------------------------
# Image preprocessing
# ------------------------------------------------------------

def preprocess_image(image):

    image = image.resize((224, 224))

    image_array = np.array(image).astype(np.float32) / 255.0

    tensor = torch.tensor(
        image_array
    ).permute(2, 0, 1)

    tensor = tensor.unsqueeze(0)

    # ImageNet normalization
    mean = torch.tensor(
        [0.485, 0.456, 0.406]
    ).view(1, 3, 1, 1)

    std = torch.tensor(
        [0.229, 0.224, 0.225]
    ).view(1, 3, 1, 1)

    tensor = (tensor - mean) / std

    return tensor.to(device)

# ------------------------------------------------------------
# Generate Grad-CAM
# ------------------------------------------------------------

def generate_gradcam(image_tensor):

    activations.clear()
    gradients.clear()

    model.zero_grad()

    output = model(image_tensor)

    predicted_class = output.argmax(dim=1).item()

    score = output[0, predicted_class]

    score.backward()

    feature_maps = activations[0]
    grads = gradients[0]

    # Global average pooling of gradients
    weights = grads.mean(
        dim=(2, 3),
        keepdim=True
    )

    cam = (weights * feature_maps).sum(
        dim=1
    )

    cam = F.relu(cam)

    cam = F.interpolate(
        cam.unsqueeze(1),
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    )

    cam = cam.squeeze().detach().cpu().numpy()

    # Normalize
    cam = cam - cam.min()

    if cam.max() > 0:
        cam = cam / cam.max()

    return cam, predicted_class

# ------------------------------------------------------------
# Plot Grad-CAM results
# ------------------------------------------------------------

fig, axes = plt.subplots(
    len(selected_files),
    3,
    figsize=(12, len(selected_files) * 4)
)

if len(selected_files) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, filename in enumerate(selected_files):

    image_path = os.path.join(
        IMAGE_DIR,
        filename
    )

    image = Image.open(
        image_path
    ).convert("RGB")

    image_tensor = preprocess_image(image)

    cam, predicted_class = generate_gradcam(
        image_tensor
    )

    true_label = incorrect_df.loc[
        incorrect_df["filename"] == filename,
        "true_label"
    ].values[0]

    predicted_label = (
        "good"
        if predicted_class == 0
        else "fail"
    )

    image_display = image.resize(
        (224, 224)
    )

    # Original image
    axes[row, 0].imshow(image_display)

    axes[row, 0].set_title(
        f"{filename}\n"
        f"True: {true_label.upper()}"
    )

    axes[row, 0].axis("off")

    # Grad-CAM
    axes[row, 1].imshow(
        cam,
        cmap="jet"
    )

    axes[row, 1].set_title(
        f"Grad-CAM\n"
        f"Predicted: {predicted_label.upper()}"
    )

    axes[row, 1].axis("off")

    # Overlay
    axes[row, 2].imshow(
        image_display
    )

    axes[row, 2].imshow(
        cam,
        cmap="jet",
        alpha=0.45
    )

    axes[row, 2].set_title(
        "Grad-CAM Overlay"
    )

    axes[row, 2].axis("off")

plt.suptitle(
    "ResNet18 — Grad-CAM Interpretability",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

gradcam_file = (
    "/content/resnet18_gradcam_analysis.png"
)

plt.savefig(
    gradcam_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Remove hooks
# ------------------------------------------------------------

forward_handle.remove()
backward_handle.remove()

print("\nSaved:")
print(gradcam_file)

print("\n RESNET18 GRAD-CAM ANALYSIS COMPLETE")

In [ ]:
# ============================================================
# DOWNLOAD METHOD 2 — RESNET18 ALL OUTPUTS
# ============================================================

import os
import shutil
from google.colab import files

OUTPUT_DIR = "/content/METHOD_2_RESNET18_OUTPUTS"

# Remove old folder if it exists
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR)

# ------------------------------------------------------------
# ResNet18 output files
# ------------------------------------------------------------

resnet_files = [
    "/content/resnet18_confusion_matrix.png",
    "/content/resnet18_100_image_predictions.csv",
    "/content/resnet18_error_analysis.csv",
    "/content/resnet18_misclassified_images.png",
    "/content/resnet18_gradcam_analysis.png",
]

# ------------------------------------------------------------
# Copy files
# ------------------------------------------------------------

for file_path in resnet_files:

    if os.path.exists(file_path):

        shutil.copy(
            file_path,
            OUTPUT_DIR
        )

        print("✓ Copied:", os.path.basename(file_path))

    else:

        print("⚠ NOT FOUND:", file_path)

# ------------------------------------------------------------
# Copy training curves
# ------------------------------------------------------------

training_curves = "/content/resnet18_training_curves"

if os.path.exists(training_curves):

    shutil.copytree(
        training_curves,
        os.path.join(
            OUTPUT_DIR,
            "resnet18_training_curves"
        )
    )

    print("✓ Copied: resnet18_training_curves")

else:

    print("⚠ Training curves folder not found")

# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------

ZIP_BASE = "/content/METHOD_2_RESNET18_OUTPUTS"

shutil.make_archive(
    ZIP_BASE,
    "zip",
    OUTPUT_DIR
)

ZIP_FILE = ZIP_BASE + ".zip"

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("METHOD 2 — RESNET18 OUTPUTS READY")
print("=" * 60)

print("\nZIP file:")
print(ZIP_FILE)

print("\n✓ Confusion matrix")
print("✓ 100-image predictions")
print("✓ Error analysis")
print("✓ Misclassified images")
print("✓ Grad-CAM")
print("✓ Training curves")

# ------------------------------------------------------------
# Download
# ------------------------------------------------------------

files.download(ZIP_FILE)

In [ ]:
# ============================================================
# DOWNLOAD METHOD 2 — RESNET18 ALL OUTPUTS
# INCLUDING 4 × 25-IMAGE RESULT SHEETS
# ============================================================

import os
import shutil
import math
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

# ------------------------------------------------------------
# FOLDERS
# ------------------------------------------------------------

OUTPUT_DIR = "/content/METHOD_2_RESNET18_OUTPUTS"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR)

IMAGE_SHEETS_DIR = os.path.join(
    OUTPUT_DIR,
    "100_IMAGE_RESULT_SHEETS"
)

os.makedirs(IMAGE_SHEETS_DIR)

# ------------------------------------------------------------
# LOAD 100-IMAGE PREDICTIONS
# ------------------------------------------------------------

prediction_file = "/content/resnet18_100_image_predictions.csv"

if not os.path.exists(prediction_file):
    raise FileNotFoundError(
        "resnet18_100_image_predictions.csv was not found."
    )

predictions_df = pd.read_csv(prediction_file)

print("=" * 60)
print("RESNET18 — 100 IMAGE RESULT SHEETS")
print("=" * 60)

print("Number of predictions:", len(predictions_df))

assert len(predictions_df) == 100, \
    "Prediction file does not contain exactly 100 images."

# ------------------------------------------------------------
# CREATE 4 SHEETS
# 25 IMAGES PER SHEET
# ------------------------------------------------------------

for sheet_number in range(4):

    start = sheet_number * 25
    end = start + 25

    subset = predictions_df.iloc[start:end]

    fig, axes = plt.subplots(
        5,
        5,
        figsize=(20, 20)
    )

    axes = axes.flatten()

    for ax, (_, row) in zip(
        axes,
        subset.iterrows()
    ):

        filename = row["filename"]

        image_path = os.path.join(
            "/content/images",
            filename
        )

        if os.path.exists(image_path):

            image = Image.open(
                image_path
            ).convert("RGB")

            ax.imshow(image)

        else:

            ax.text(
                0.5,
                0.5,
                "IMAGE NOT FOUND",
                ha="center",
                va="center"
            )

        true_label = str(
            row["true_label"]
        ).upper()

        predicted_label = str(
            row["predicted_label"]
        ).upper()

        correct = row["correct"]

        if isinstance(correct, str):
            correct = correct.lower() == "true"

        symbol = "✓" if correct else "✗"

        ax.set_title(
            f"{filename}\n"
            f"TRUE: {true_label} | "
            f"PRED: {predicted_label} {symbol}",
            fontsize=9
        )

        ax.axis("off")

    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    fig.suptitle(
        f"ResNet18 — 100 Image Out-of-Fold Results "
        f"({start + 1}–{end})",
        fontsize=18,
        fontweight="bold"
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_file = os.path.join(
        IMAGE_SHEETS_DIR,
        f"resnet18_results_{start + 1}_{end}.png"
    )

    plt.savefig(
        output_file,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print(
        f"✓ Saved: {output_file}"
    )

# ------------------------------------------------------------
# COPY OTHER RESNET18 OUTPUTS
# ------------------------------------------------------------

resnet_files = [

    "/content/resnet18_confusion_matrix.png",

    "/content/resnet18_100_image_predictions.csv",

    "/content/resnet18_error_analysis.csv",

    "/content/resnet18_misclassified_images.png",

    "/content/resnet18_gradcam_analysis.png",
]

for file_path in resnet_files:

    if os.path.exists(file_path):

        shutil.copy(
            file_path,
            OUTPUT_DIR
        )

        print(
            "✓ Copied:",
            os.path.basename(file_path)
        )

    else:

        print(
            "⚠ NOT FOUND:",
            file_path
        )

# ------------------------------------------------------------
# COPY TRAINING CURVES
# ------------------------------------------------------------

training_curves = "/content/resnet18_training_curves"

if os.path.exists(training_curves):

    shutil.copytree(
        training_curves,
        os.path.join(
            OUTPUT_DIR,
            "resnet18_training_curves"
        )
    )

    print(
        "✓ Copied: resnet18_training_curves"
    )

else:

    print(
        "⚠ Training curves folder not found"
    )

# ------------------------------------------------------------
# CREATE ZIP
# ------------------------------------------------------------

ZIP_BASE = "/content/METHOD_2_RESNET18_OUTPUTS"

shutil.make_archive(
    ZIP_BASE,
    "zip",
    OUTPUT_DIR
)

ZIP_FILE = ZIP_BASE + ".zip"

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("METHOD 2 — RESNET18 COMPLETE OUTPUT PACKAGE")
print("=" * 60)

print("""
Included:

✓ Confusion matrix
✓ 100-image prediction CSV
✓ 100-image results — Sheet 1 (1–25)
✓ 100-image results — Sheet 2 (26–50)
✓ 100-image results — Sheet 3 (51–75)
✓ 100-image results — Sheet 4 (76–100)
✓ Error analysis CSV
✓ Misclassified image visualisation
✓ Grad-CAM analysis
✓ Training/validation accuracy curve
✓ Training loss curve
""")

print("ZIP file:")
print(ZIP_FILE)

# ------------------------------------------------------------
# DOWNLOAD
# ------------------------------------------------------------

files.download(ZIP_FILE)

In [ ]:
# ============================================================
# METHOD 1 — FINAL MODEL COMPARISON
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Final results from 5-fold out-of-fold evaluation
# ------------------------------------------------------------

method1_comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting",
        "SVM"
    ],
    "Accuracy": [
        0.780,
        0.800,
        0.780
    ],
    "Precision": [
        0.811,
        0.804,
        0.848
    ],
    "Recall": [
        0.667,
        0.792,
        0.622
    ],
    "Macro-F1": [
        0.773,
        0.795,
        0.769
    ]
})

print("=" * 70)
print("METHOD 1 — FINAL MODEL COMPARISON")
print("=" * 70)

display(method1_comparison)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

method1_comparison.to_csv(
    "/content/method1_model_comparison.csv",
    index=False
)

print("\nSaved:")
print("/content/method1_model_comparison.csv")

# ------------------------------------------------------------
# Comparison graph
# ------------------------------------------------------------

metrics = ["Accuracy", "Precision", "Recall", "Macro-F1"]

x = np.arange(len(method1_comparison["Model"]))
width = 0.18

plt.figure(figsize=(12, 6))

for i, metric in enumerate(metrics):
    plt.bar(
        x + (i - 1.5) * width,
        method1_comparison[metric],
        width,
        label=metric
    )

plt.xticks(
    x,
    method1_comparison["Model"]
)

plt.ylabel("Score")
plt.xlabel("Classification Model")
plt.ylim(0, 1.0)

plt.title(
    "Method 1 — Comparison of Classical Machine-Learning Models"
)

plt.legend()
plt.grid(axis="y", alpha=0.25)

plt.tight_layout()

plt.savefig(
    "/content/method1_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print("/content/method1_model_comparison.png")

In [ ]:
# ============================================================
# FINAL METHOD 1 vs METHOD 2 COMPARISON
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Enter the verified final results
# ------------------------------------------------------------

comparison = pd.DataFrame({

    "Method": [
        "Method 1 - Gradient Boosting",
        "Method 2 - ResNet18"
    ],

    "Accuracy": [
        0.800,
        0.820
    ],

    "Precision": [
        0.804,
        0.865
    ],

    "Recall": [
        0.792,
        0.711
    ],

    "Macro-F1": [
        0.795,
        0.814
    ]
})

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("METHOD 1 vs METHOD 2 — FINAL COMPARISON")
print("=" * 70)

display(
    comparison.style.format({
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "Macro-F1": "{:.3f}"
    })
)

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

comparison_csv = "/content/method1_vs_method2_comparison.csv"

comparison.to_csv(
    comparison_csv,
    index=False
)

print("\n✓ Comparison table created")
print("✓ Saved:", comparison_csv)

In [ ]:
# ============================================================
# METHOD 1 vs METHOD 2 — PERFORMANCE COMPARISON GRAPH
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "Macro-F1"
]

method1 = [
    0.800,
    0.804,
    0.792,
    0.795
]

method2 = [
    0.820,
    0.865,
    0.711,
    0.814
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 7))

bars1 = ax.bar(
    x - width/2,
    method1,
    width,
    label="Method 1 — Gradient Boosting"
)

bars2 = ax.bar(
    x + width/2,
    method2,
    width,
    label="Method 2 — ResNet18"
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax.set_xlabel("Performance Metric", fontsize=12)
ax.set_ylabel("Score", fontsize=12)

ax.set_title(
    "Method 1 vs Method 2 Performance Comparison",
    fontsize=16,
    fontweight="bold"
)

ax.set_xticks(x)
ax.set_xticklabels(metrics)

ax.set_ylim(0, 1.0)

ax.legend()

# ------------------------------------------------------------
# Values above bars
# ------------------------------------------------------------

for bar, value in zip(bars1, method1):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

for bar, value in zip(bars2, method2):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

comparison_plot = (
    "/content/method1_vs_method2_performance_comparison.png"
)

plt.savefig(
    comparison_plot,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("=" * 70)
print("METHOD 1 vs METHOD 2 — GRAPH COMPLETE")
print("=" * 70)
print("Saved:", comparison_plot)

In [ ]:
# ============================================================
# METHOD 1 vs METHOD 2 — CONFUSION MATRIX COMPARISON
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Final verified confusion matrices
# Rows    = True label
# Columns = Predicted label
#
# 0 = Good
# 1 = Fail
# ------------------------------------------------------------

cm_method1 = np.array([
    [48, 7],
    [13, 32]
])

cm_method2 = np.array([
    [50, 5],
    [13, 32]
])

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

# ============================================================
# METHOD 1
# ============================================================

im1 = axes[0].imshow(cm_method1)

axes[0].set_title(
    "Method 1 — Gradient Boosting",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])

axes[0].set_xticklabels(["Good", "Fail"])
axes[0].set_yticklabels(["Good", "Fail"])

for i in range(2):
    for j in range(2):
        axes[0].text(
            j,
            i,
            cm_method1[i, j],
            ha="center",
            va="center",
            fontsize=16
        )

# ============================================================
# METHOD 2
# ============================================================

im2 = axes[1].imshow(cm_method2)

axes[1].set_title(
    "Method 2 — ResNet18",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])

axes[1].set_xticklabels(["Good", "Fail"])
axes[1].set_yticklabels(["Good", "Fail"])

for i in range(2):
    for j in range(2):
        axes[1].text(
            j,
            i,
            cm_method2[i, j],
            ha="center",
            va="center",
            fontsize=16
        )

plt.suptitle(
    "Confusion Matrix Comparison",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

cm_comparison_path = (
    "/content/method1_vs_method2_confusion_matrices.png"
)

plt.savefig(
    cm_comparison_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("=" * 70)
print("CONFUSION MATRIX COMPARISON COMPLETE")
print("=" * 70)
print("Saved:", cm_comparison_path)

In [ ]:
# ============================================================
# DOWNLOAD METHOD 1 vs METHOD 2 COMPARISON OUTPUTS
# ============================================================

from google.colab import files
import zipfile
import os

files_to_download = [
    "/content/method1_vs_method2_performance_comparison.png",
    "/content/method1_vs_method2_confusion_matrices.png",
    "/content/method1_vs_method2_comparison.csv"
]

zip_path = "/content/method1_vs_method2_comparison_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    for file_path in files_to_download:

        if os.path.exists(file_path):
            zipf.write(
                file_path,
                os.path.basename(file_path)
            )
            print("Added:", os.path.basename(file_path))
        else:
            print("NOT FOUND:", file_path)

print("\n" + "=" * 60)
print("DOWNLOAD READY")
print("=" * 60)
print("ZIP:", zip_path)

files.download(zip_path)

In [ ]:
# ============================================================
# FINAL TABLE — METHOD 1 vs METHOD 2
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# VERIFIED RESULTS
# ------------------------------------------------------------

final_comparison = pd.DataFrame({
    "Method": [
        "Method 1 - Gradient Boosting",
        "Method 2 - ResNet18"
    ],

    "Accuracy": [
        0.800,
        0.820
    ],

    "Precision": [
        0.804,
        0.865
    ],

    "Recall": [
        0.792,
        0.711
    ],

    "F1-Score": [
        0.795,
        0.780
    ],

    "Macro-F1": [
        0.795,
        0.814
    ],

    "Correct": [
        80,
        82
    ],

    "Incorrect": [
        20,
        18
    ],

    "Good-to-Fail": [
        7,
        5
    ],

    "Fail-to-Good": [
        13,
        13
    ],

    "Error Rate": [
        0.200,
        0.180
    ]
})

# ------------------------------------------------------------
# DISPLAY AS PERCENTAGES WHERE APPROPRIATE
# ------------------------------------------------------------

display_table = final_comparison.copy()

percentage_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "Macro-F1",
    "Error Rate"
]

for col in percentage_columns:
    display_table[col] = (
        display_table[col] * 100
    ).round(1).astype(str) + "%"

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 100)
print("FINAL COMPARISON — METHOD 1 vs METHOD 2")
print("=" * 100)

display(display_table)

# ------------------------------------------------------------
# SAVE CSV
# ------------------------------------------------------------

csv_path = "/content/final_method1_vs_method2_comparison.csv"

final_comparison.to_csv(
    csv_path,
    index=False
)

print("\nSaved:")
print(csv_path)